In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-15 03:44:35--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.4’

rag_helper.py.4     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-15 03:44:35 (8.37 MB/s) - ‘rag_helper.py.4’ saved [2134/2134]

--2026-06-15 03:44:35--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 20

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [4]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Create the assitant

In [5]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [6]:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about running **Olama** locally in the provided context.\n\nThe closest related item is about running the **MCP Inspector** locally:\n\n```bash\nnpx @modelcontextprotocol/inspector\n```\n\nIf you meant something else by “Olama,” please share the exact tool name or more context.'

In [7]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure it out. Please send:\n- the course name or link\n- where it’s hosted\n- whether it’s live, self-paced, or already in progress\n\nIf you’re asking generally, the best next step is to check:\n- the enrollment deadline\n- whether late registration is allowed\n- any prerequisites or approval needed\n\nIf you’d like, I can also help you draft a quick message to the instructor or organizer asking if you can still join.'

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search('How do I run Olama locally?')

[{'id': '42c7af7ed5',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Agents',
  'question': 'Run MCP Inspector',
  'answer': 'To run the MCP Inspector, execute the following command in the terminal:\n\n```bash\nnpx @modelcontextprotocol/inspector\n```'},
 {'id': 'e394e6f738',
  'course': 'llm-zoomcamp',
  'section': 'Workshop: Open-Source Data Ingestion (dlt)',
  'question': 'How do I know which tables are in the db?',
  'answer': 'You can use the `db.table_names()` method to list all the tables in the database.'},
 {'id': '9b0e2d6a47',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Project: how do I evaluate a recommender-style RAG (no obvious Q&A ground truth)?',
  'answer': "Two complementary approaches that both score for the evaluation criterion:\n\n1. Synthetic ground truth (same idea as the course, adapted). For each item in your dataset, prompt the LLM with the item's description and ask it to generate ~5 user queries that should return that it

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [11]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course join late enrollment"}', call_id='call_4oxifL9Wm2DT4sWRA0lR4cwF', name='search', type='function_call', id='fc_02e0fd3d43f6123d006a2f7536d6a8819b90cf096431393109', namespace=None, status='completed')]

In [12]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [13]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [14]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [15]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(657, 31)

In [16]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


A Developer Prompt

In [17]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [18]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [19]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join the course late enrollment discover course can I join it"}
function_call: search {"query":"course enrollment registration when can I join the course"}
function_call: search {"query":"late join course FAQ discovered the course can I join"}


In [20]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open.

Also, you don’t need a confirmation email to start learning or submit homework while the form is open.

Would you like to know how the certificate or homework submission process works?


In [21]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [22]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama locally run install local model FAQ"}
iteration #2...
function_call: search {"query":"Ollama serve localhost 11434 Python client ollama run llama3 local server"}
iteration #3...
ASSISTANT:
To run Ollama locally:

1. Install Ollama from: https://ollama.com/download  
   - macOS: install the `.pkg`
   - Windows: install the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a model locally:
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat interface.

3. Check that the local server is running:
   ```bash
   curl http://localhost:11434
   ```

4. If you want to use it from Python:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello"}]
   )

   print(response['message']['content'])
   ```

If Ollama isn’t respondin

'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download  \n   - macOS: install the `.pkg`\n   - Windows: install the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Start a model locally:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat interface.\n\n3. Check that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. If you want to use it from Python:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf Ollama isn’t responding, you can restart the server with:\n```bash\nollama serve\n```\n\nIf you want, I can also show you how to use Ollama in a notebook or connect it to a RAG app.'

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered late can I join enrollment FAQ"}
iteration #2...
function_call: search {"query":"certificate submit project while still accepting submissions live cohort self-paced FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while the course is still accepting submissions, and you’ll also need to complete the peer-review requirements during the live cohort.

If you’d like, I can also explain the certificate rules or how to catch up if you’re joining late.


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while the course is still accepting submissions, and you’ll also need to complete the peer-review requirements during the live cohort.\n\nIf you’d like, I can also explain the certificate rules or how to catch up if you’re joining late.'

In [24]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit what is queen gambit chess opening"}
iteration #2...
function_call: search {"query":"chess opening queen's gambit queen gambit definition"}
iteration #3...
ASSISTANT:
A **Queen’s Gambit** is a **chess opening**.

It starts with:
1. **d4 d5**
2. **c4**

The idea is that White offers the **c-pawn** as a “gambit” to try to gain control of the center of the board.

There are a few common variations, like:
- **Queen’s Gambit Accepted**: Black takes the pawn on c4
- **Queen’s Gambit Declined**: Black does not take it

If you meant the **Netflix show** *The Queen’s Gambit*, I can explain that too. Want the chess version or the show?


'A **Queen’s Gambit** is a **chess opening**.\n\nIt starts with:\n1. **d4 d5**\n2. **c4**\n\nThe idea is that White offers the **c-pawn** as a “gambit” to try to gain control of the center of the board.\n\nThere are a few common variations, like:\n- **Queen’s Gambit Accepted**: Black takes the pawn on c4\n- **Queen’s Gambit Declined**: Black does not take it\n\nIf you meant the **Netflix show** *The Queen’s Gambit*, I can explain that too. Want the chess version or the show?'

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen’s gambit,” so it looks like this is probably off-topic for the course.

If you meant something specific in the course context, feel free to rephrase it. Are there other areas of the course or logistics you want to explore?


'I couldn’t find any course FAQ entry about “queen’s gambit,” so it looks like this is probably off-topic for the course.\n\nIf you meant something specific in the course context, feel free to rephrase it. Are there other areas of the course or logistics you want to explore?'

In [26]:
!uv add toyaikit

Resolved 127 packages in 4.76s                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/18.22 KiB           
⠙ Preparing packages... (0/7)---------- 14.83 KiB/18.22 KiB         
⠙ Preparing packages... (0/7)---------- 14.83 KiB/18.22 KiB         
truststore           ------------------------------ 14.83 KiB/18.22 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
truststore           ------------------------------ 14.83 KiB/18.22 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
truststore           ------------------------------ 14.83 KiB/18.22 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
truststore           ------------------------------ 14.83 KiB/18.22 KiB
docstring-parser     ----------

In [27]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [28]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [29]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [30]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [31]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]